# 01 · Preparação e qualidade dos dados

**Objetivo:** preparar o painel mensal para estudar a evolução da inflação alimentar e a sua associação com o clima e a estrutura económica.

**Percurso:** perguntas → unidades → formato longo → cobertura → omissos e extremos → conferência das transformações. As funções encontram-se em `scripts/preparacao.py`; este notebook mostra como são usadas e interpreta os resultados.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Permite executar o notebook a partir de Trabalho/ ou de notebooks/.
pasta_atual = Path.cwd().resolve()
raiz = pasta_atual if (pasta_atual / "scripts").is_dir() else pasta_atual.parent
assert (raiz / "scripts").is_dir(), "Abra o Jupyter na pasta Trabalho ou em notebooks/."
if str(raiz / "scripts") not in sys.path:
    sys.path.insert(0, str(raiz / "scripts"))

from preparacao import ler_painel
from estilo_iscte import (
    configurar,
    capa,
    indicadores,
    tabela,
    nota,
    csv,
    dirs,
    pt,
    Y,
    VARS,
    VAR_LABELS,
    UNITS,
)

configurar()
pasta_dados = raiz / "dados" / "originais"
dados = ler_painel(pasta_dados)

from preparacao import (
    resumir_cobertura,
    contar_omissos,
    conferir_campos,
    conferir_transformacoes,
    resumir_extremos,
)

tabelas, _ = dirs(raiz, "01_preparacao")
resumo, cobertura = resumir_cobertura(dados)
capa(
    "01",
    "Preparação e qualidade",
    "Perguntas, unidades e verificações de integridade do painel mensal.",
)
indicadores(
    [
        ("Observações", pt(resumo["linhas"], 0)),
        ("Países", resumo["paises"]),
        ("Meses por país", resumo["meses"]),
        ("Período", "02/2000–12/2025"),
    ]
)

## 1. Questões de investigação

**Q1 — Evolução:** como evoluiu a inflação alimentar mensal entre fevereiro de 2000 e dezembro de 2025, e em que diferem as trajetórias dos 15 países?

**Q2 — Clima e contexto económico:** a inflação alimentar associa-se às anomalias térmicas e ao défice hídrico? Poderá essa associação variar com o desenvolvimento económico ou o peso da agricultura?

A resposta mede a variação mensal de preços dos alimentos. As variáveis climáticas comparam o mês com uma referência habitual de cada país. O PIB por habitante e o peso agrícola caracterizam contextos distintos; as interações permitem investigar diferenças de associação. O PIB não mede diretamente o rendimento disponível das famílias. Os dados observacionais, sem controlo de fatores omitidos, não permitem atribuir causalidade.

## 2. Variáveis, unidades e frequência

A resposta e as variáveis climáticas são mensais. As covariáveis económicas são anuais, referentes ao ano anterior, e repetem-se nos meses do ano corrente. Não houve interpolação. A repetição não cria medições económicas mensais independentes.

A inflação é `100 × [ln(IPC_t) − ln(IPC_t−1)]`: aproximadamente percentagem mensal, diferente da taxa homóloga e da variação percentual exata. O défice hídrico positivo indica condições mais secas; não é SPEI. As interações têm unidades compostas.

In [2]:
unidades = pd.DataFrame(
    [
        {"Variável": VAR_LABELS[coluna], "Coluna": coluna, "Unidade": UNITS[coluna]}
        for coluna in VARS
    ]
)
tabela(unidades, "Tabela 1.1 · Variáveis analíticas e unidades")
csv(unidades, tabelas / "01_variaveis_e_unidades.csv")

Variável,Coluna,Unidade
Inflação alimentar,inflacao_alimentar_log_pct,100 × Δ ln IPC; aprox. %/mês
Anomalia térmica,anomalia_temperatura_c,°C
Défice hídrico de 3 meses,defice_hidrico_3m_z,desvios padrão
Log PIB real por habitante (t−1),ln_pib_pc_ppc_ano_anterior,log natural
Peso agrícola no PIB (t−1),peso_agricola_pct_ano_anterior,pontos percentuais do PIB
Calor × log PIB,calor_x_ln_pib,°C × log PIB
Calor × peso agrícola,calor_x_peso_agricola,°C × p.p. do PIB


## 3. Formato longo e cobertura

Cada linha representa um país num mês. A chave é `pais_iso3` + `data`. O primeiro dia do mês identifica o período, não a data exata de recolha. Verificamos os pares da grelha completa país × mês, além de contar as observações por país.

In [3]:
conferir_campos(dados, VARS)
assert resumo["duplicados"] == 0
assert resumo["equilibrado"]

cobertura_visivel = cobertura.rename(
    columns={
        "pais_iso3": "ISO",
        "pais": "País",
        "meses": "Meses",
        "inicio": "Início",
        "fim": "Fim",
    }
)
for coluna in ["Início", "Fim"]:
    cobertura_visivel[coluna] = cobertura_visivel[coluna].dt.strftime("%m/%Y")

tabela(cobertura_visivel, "Tabela 1.2 · Cobertura por país", precisao=0)
csv(cobertura, tabelas / "02_cobertura_por_pais.csv")
csv(dados, raiz / "dados" / "processados" / "painel_longo.csv")

ISO,País,Meses,Início,Fim
BDI,Burundi,311,02/2000,12/2025
BEN,Benim,311,02/2000,12/2025
BFA,Burkina Faso,311,02/2000,12/2025
BRA,Brasil,311,02/2000,12/2025
ESP,Espanha,311,02/2000,12/2025
HND,Honduras,311,02/2000,12/2025
JPN,Japão,311,02/2000,12/2025
MEX,México,311,02/2000,12/2025
MLI,Mali,311,02/2000,12/2025
NER,Níger,311,02/2000,12/2025


In [4]:
nota(
    "Diagnóstico · painel equilibrado",
    f"O painel tem {pt(resumo['linhas'], 0)} linhas e {resumo['colunas']} colunas, "
    f"com {resumo['paises']} países e {resumo['meses']} meses comuns. "
    f"Foram encontrados {resumo['duplicados']} duplicados e "
    f"{resumo['pares_ausentes']} pares país–mês ausentes. "
    "Janeiro de 2000 fica apenas nos dados brutos, pois é necessário para calcular a primeira variação do IPC.",
)

## 4. Omissos, valores extremos e decisões de tratamento

Inspecionamos as 16 colunas. Não se substituem valores em falta por zero. Os extremos são sinalizados pela regra `Q1 − 1,5 × IQR` e `Q3 + 1,5 × IQR`; essa regra pode assinalar diferenças estruturais entre países e não determina exclusões.

In [5]:
omissos = contar_omissos(dados)
tabela(omissos, "Tabela 1.3 · Valores em falta por coluna")
csv(omissos, tabelas / "03_omissos_e_tipos.csv")

extremos = resumir_extremos(dados, VARS)
extremos.insert(0, "Variável", extremos.coluna.map(VAR_LABELS))
tabela(extremos.drop(columns="coluna"), "Tabela 1.4 · Sinalização IQR global")
csv(extremos, tabelas / "04_sinalizacao_iqr.csv")

Coluna,Tipo,Omissos,Omissos (%)
pais_iso3,object,"0,000","0,000"
pais,object,"0,000","0,000"
data,datetime64[ns],"0,000","0,000"
ano,int64,"0,000","0,000"
mes,int64,"0,000","0,000"
inflacao_alimentar_log_pct,float64,"0,000","0,000"
anomalia_temperatura_c,float64,"0,000","0,000"
defice_hidrico_3m_z,float64,"0,000","0,000"
ln_pib_pc_ppc_ano_anterior,float64,"0,000","0,000"
peso_agricola_pct_ano_anterior,float64,"0,000","0,000"


Variável,Sinalizados,Percentagem
Inflação alimentar,"519,000","11,125"
Anomalia térmica,"156,000","3,344"
Défice hídrico de 3 meses,"53,000","1,136"
Log PIB real por habitante (t−1),"0,000","0,000"
Peso agrícola no PIB (t−1),"0,000","0,000"
Calor × log PIB,"227,000","4,866"
Calor × peso agrícola,"781,000","16,742"


In [6]:
nota(
    "Decisão · preservar a amostra",
    f"Existem {int(omissos.Omissos.sum())} valores em falta. "
    "Não foi necessária imputação, e não foram aplicadas exclusões ou winsorização. "
    "Os valores extremos são analisados por país e conferidos com o IPC de origem no notebook 03. "
    "A cópia processada mantém os valores da entrada, apenas com ordenação cronológica.",
)

## 5. Conferência das transformações e do alinhamento anual

Verificamos o logaritmo do PIB, o peso agrícola do ano anterior e as duas interações usando a tabela anual. Uma concordância numérica verifica o cálculo; não demonstra que o dado já estivesse publicado em tempo real.

In [7]:
conferencia = conferir_transformacoes(dados, pasta_dados)
tabela(
    conferencia,
    "Tabela 1.5 · Conferência das transformações",
    formatos={"Erro absoluto máximo": lambda valor: f"{valor:.2e}"},
)
csv(conferencia, tabelas / "05_transformacoes_conferidas.csv")

Verificação,Erro absoluto máximo
Log PIB do ano anterior,1.78e-15
Peso agrícola do ano anterior,0.00e+00
Calor × log PIB,1.42e-14
Calor × peso agrícola,1.42e-14


## 6. Diagnóstico e alterações necessárias

**Manter:** a frequência mensal, o formato longo e a amostra integral. A tabela anual confirma o alinhamento económico e os cálculos das interações.

**Explicitar no texto do trabalho:** covariáveis económicas anuais desfasadas; inflação logarítmica mensal; défice hídrico padronizado por país e mês; classificação de rendimento atual. A inexistência de omissos não prova qualidade homogénea das fontes.

**Investigar a seguir:** distribuições e trajetórias no notebook 02; sazonalidade, dependência temporal, choques comuns e extremos no notebook 03. As fontes, versões e regras estão em `dados/originais/fontes.csv` e `metodologia_original.txt`.

In [8]:
import json
from estilo_iscte import hash_file

registo = {
    **resumo,
    "omissos": int(omissos.Omissos.sum()),
    "sha256_entrada": hash_file(pasta_dados / "painel_indicadores.csv"),
}
(tabelas / "06_registo_qualidade.json").write_text(
    json.dumps(registo, ensure_ascii=False, indent=2), encoding="utf-8"
)
indicadores(
    [
        ("Duplicados", resumo["duplicados"]),
        ("Pares ausentes", resumo["pares_ausentes"]),
        ("Omissos", registo["omissos"]),
        ("Imputações", 0),
    ]
)